<a href="https://colab.research.google.com/github/radionovakd/compling-hw/blob/main/%D0%A0%D0%B0%D0%B4%D0%B8%D0%BE%D0%BD%D0%BE%D0%B2%D0%B0_RAG_LAngchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Тема:** RAG (Retrieval-Augmented Generation) с фреймворком LangChain


Разработка RAG-пайплайна


**Задачи:**
* Загрузить набор текстовых документов (например, статей из датасета arXiv Dataset: https://www.kaggle.com/datasets/Cornell-University/arxiv)
* Разбить текст на чанки с помощью Langchain text splitter
* Создать векторный индекс с помощью FAISS и sentence-transformers
* Реализовать langchain-цепочку, которая производим семантический поиск и формирует промпт для LLM (локальной или через Groq/OpenRouter)
* Протестировать систему на нескольких вопросах, оценить качество ответов


**Библиотеки:** langchain, huggingface, faiss-cpu, sentence-transformers

**Ожидаемый результат:** Colab-ноутбук с рабочим прототипом наукоёмкой (например, разработанной на основе текстов ArXiv) RAG-системы, примерами её ответов и качественным анализом, представленным в текстовых блоках


## Загрузить набор текстовых документов

### Датасет с метаданными к статьям

В качестве данных я решила взять библейские тексты - чтоб было интерсно и необычно + люблю их. Что из этого выйдет - узнаем на выходе.

In [ ]:

import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "t_bbe.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "oswinrh/bible",
  file_path,
)

print(df.head())
# импорт - тут всё примитивно, получили данные в формате датафрейм

/tmp/ipykernel_82155/4183683005.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'bible' dataset.
        id  b  c  v                                                  t
0  1001001  1  1  1    At the first God made the heaven and the earth.
1  1001002  1  1  2  And the earth was waste and without form; and ...
2  1001003  1  1  3  And God said, Let there be light: and there wa...
3  1001004  1  1  4  And God, looking on the light, saw that it was...
4  1001005  1  1  5  Naming the light, Day, and the dark, Night. An...


In [ ]:
print(df.dtypes)
# просто посмотреть чё вообще происходит с данными данными:)

id     int64
b      int64
c      int64
v      int64
t     object
dtype: object


### Загрузка

In [ ]:
!pip install langchain_community langchain_text_splitters pypdf -q

In [ ]:
pip install langchain pandas

In [ ]:
from langchain_community.document_loaders import DataFrameLoader

In [ ]:
loader = DataFrameLoader(df, page_content_column='t')
documents = loader.load()
# скачали df - им же и пользуемся в преобразовании метаданных с текстом в документ

In [ ]:
print(documents[48])
# опять же смотрим, что происходит, а происходит крутое - нехорошо человеку быть одному!

page_content='And the Lord God said, It is not good for the man to be by himself: I will make one like himself as a help to him' metadata={'id': 1002018, 'b': 1, 'c': 2, 'v': 18}


## Разбить на чанки

In [ ]:
from langchain_core.documents import Document
from collections import defaultdict


Здесь начинаются все прелести выбранного мною датасета. Файл выстроен таким образом, что каждый стих - отдельный документ. Возможно это и сошло бы за чанк, но он остаётся слишком маленьким для такого большого количества документов - 31101, оказывается это реальное количество стихов в Библии. Чтобы интерснее разбить всю эту историю, мы их сгруппируем по главам, а затем уже разобьём на чанки с помомщью оптимального сплитера.

In [ ]:
grouped = defaultdict(list)

for d in documents:
    # группируем по главе
    key = (d.metadata["b"], d.metadata["c"])
    grouped[key].append(d)

chapter_docs = []

for (book, chapter), verses in grouped.items():
    text = "\n".join([v.page_content for v in verses])

    chapter_docs.append(
        Document(
            page_content=text,
            metadata={
                "book": book,
                "chapter": chapter
            }
        )
    )

In [ ]:
# Сплиттер (ОПТИМАЛЬНЫЙ)
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     # для коротких текстов неплохо
    chunk_overlap=100,  # чтобы не терять контекст
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = splitter.split_documents(chapter_docs)

print(len(chunks))
# voilà

6384


In [ ]:
print("📄 Number of Chunks:", len(chunks))

# Выводим только первые 3 чанка
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}:\n{chunk}")
    # вот и посмотрим, что у нас тут получилось такое

📄 Number of Chunks: 6384

Chunk 1:
page_content='At the first God made the heaven and the earth.
And the earth was waste and without form; and it was dark on the face of the deep: and the Spirit of God was moving on the face of the waters.
And God said, Let there be light: and there was light.
And God, looking on the light, saw that it was good: and God made a division between the light and the dark,
Naming the light, Day, and the dark, Night. And there was evening and there was morning, the first day.
And God said, Let there be a solid arch stretching over the waters, parting the waters from the waters.
And God made the arch for a division between the waters which were under the arch and those which were over it: and it was so.
And God gave the arch the name of Heaven. And there was evening and there was morning, the second day.' metadata={'book': 1, 'chapter': 1}

Chunk 2:
page_content='And God said, Let the waters under the heaven come together in one place, and let the dry land be 

## Создать векторный индекс с помощью faiss-cpu и sentence-transformers

In [ ]:
!pip install faiss-cpu sentence-transformers langchain-huggingface -q

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",  #ну это база, собственно
    encode_kwargs={"normalize_embeddings": True} #так, для мнимой точности
)

vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"Обработано чанков: {vectorstore.index.ntotal}") #было долго, но она сделала это

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Обработано чанков: 6384


## Реализовать цепочку

In [ ]:
pip install langchain-classic

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

После невроятных приключений и поиска подходящих моделей на опенроутер, было предпринято решение польностью отдаться в руки искусственного интеллекта и не просить каждый раз "в чём же прикол ошибки 404, 403, 402" - вашему вниманию, КВЕН.:

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2
)

llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_82155/195368600.py:18: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:
TOP_K = 5
# Неплохая длина для подобного формата текстов
PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant that answers questions about the Bible.

Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}

Answer (include verse reference if possible):
""")
# Каковы данные - таковы и языки, как говорила я в 4 часа утра, пытаясь придумать дельные комментарии
QUESTION = "What did Noah do?"
#вопросы ковчега

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

In [ ]:
result = qa_chain.invoke(QUESTION)

print("=== ОТВЕТ ===")
print(result["result"])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== ОТВЕТ ===

You are a helpful assistant that answers questions about the Bible.

Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
And after seven days more, he sent the dove out again, but she did not come back to him.
And in the six hundred and first year, on the first day of the first month, the waters were dry on the earth: and Noah took the cover off the ark and saw that the face of the earth was dry.
And on the twenty-seventh day of the second month the earth was dry.
And God said to Noah,
Go out of the ark, you and your wife and your sons and your sons' wives.
Take out with you every living thing which is with you, birds and cattle and everything which goes on the earth, so that they may have offspring and be fertile and be increased on the earth.
And Noah went out with his sons and his wife and his sons' wives;

And God kept Noah in mind, and all the living things and the cattle which were with him in the ark

Ответ убил - по другому не скажешь. Я ждала очевидного прикола, но он неплохо прокачался в богословии. Но, что важно - я взяла только 5 чанков, этого не хватило, чтоб мне сказали, что Ной так то ещё сам и построил этот ковчег. Вполне вероятно, что можно было б взять немного больше, но всё же посмотрим, что он на другие вопросы скажет - и это был фактический вопрос.

In [ ]:
print("Чанки ответа")

for i, doc in enumerate(result["source_documents"]):
    print(f"[Чанк {i+1}]")
    print(doc.page_content)
    print("-"*50)

Чанки ответа
[Чанк 1]
And after seven days more, he sent the dove out again, but she did not come back to him.
And in the six hundred and first year, on the first day of the first month, the waters were dry on the earth: and Noah took the cover off the ark and saw that the face of the earth was dry.
And on the twenty-seventh day of the second month the earth was dry.
And God said to Noah,
Go out of the ark, you and your wife and your sons and your sons' wives.
Take out with you every living thing which is with you, birds and cattle and everything which goes on the earth, so that they may have offspring and be fertile and be increased on the earth.
And Noah went out with his sons and his wife and his sons' wives;
--------------------------------------------------
[Чанк 2]
And God kept Noah in mind, and all the living things and the cattle which were with him in the ark: and God sent a wind over the earth, and the waters went down.
And the fountains of the deep and the windows of heaven 

Данные места Писания соотвествуют тому, о чём спрашивалось.

In [ ]:
q1 = "Who is Christina?"
res1 = qa_chain.invoke(q1)

print(res1["result"])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a helpful assistant that answers questions about the Bible.

Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
Now Dinah, the daughter whom Leah had by Jacob, went out to see the women of that country.
And when Shechem, the son of Hamor the Hivite who was the chief of that land, saw her, he took her by force and had connection with her.
Then his heart went out in love to Dinah, the daughter of Jacob, and he said comforting words to her.
And Shechem said to Hamor, his father, Get me this girl for my wife.
Now Jacob had word of what Shechem had done to his daughter; but his sons were in the fields with the cattle, and Jacob said nothing till they came.
Then Hamor, the father of Shechem, came out to have a talk with Jacob.

Those who are against her have become the head, everything goes well for her haters; for the Lord has sent sorrow on her because of the great number of her sins: her young children have gone aw

Аяяй, не попался! Вполне поянтно, что меня родители назвали следуя древнегреческому слову, однако так не только я называюсь. Хотелось подловить, как же близко моё имя стоит рядом со словом христианин. Вот что важно - тексты он всё равно дал, но преимущественно описывающие женский род - интересно, потому что в Библии в соотношении гендера, довольно сильная разница.

In [ ]:
q2 = "Who is christian?"
res2 = qa_chain.invoke(q2)

print(res2["result"])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a helpful assistant that answers questions about the Bible.

Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
Paul, an Apostle of Jesus Christ by the purpose of God, in the hope of the life which is in Christ Jesus,
To Timothy, my well-loved child: Grace, mercy, peace, from God the Father and Christ Jesus our Lord.
I give praise to God, whose servant I have been, with a heart free from sin, from the time of my fathers, because in my prayers at all times the thought of you is with me, night and day
Desiring to see you, keeping in my memory your weeping, so that I may be full of joy;
Having in mind your true faith, which first was in your mother's mother Lois, and in your mother Eunice, and, I am certain, is now in you.
For this reason I say to you, Let that grace of God which is in you, given to you by my hands, have living power.

Paul, a servant of Jesus Christ, an Apostle by the selection of God, given autho

Значит только Павел у нас христианин, а остальные так... Но, важно отметить - христианин довольно позднее понятие, в Библии не предусмотрено место этому слову, это можно заметить по чанкам - преимущественно Христос и преимущественно рядом с Павлом - тем, кто достаточно много писал о своёv учителе и в целом новый завет - approved.

In [ ]:
q3 = "Does love envy and selfish?"
res3 = qa_chain.invoke(q3)

print(res3["result"])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a helpful assistant that answers questions about the Bible.

Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
One hand full of rest is better than two hands full of trouble and desire for wind.
Then I came back, and I saw an example of what is to no purpose under the sun.
It is one who is by himself, without a second, and without son or brother; but there is no end to all his work, and he has never enough of wealth. For whom, then, am I working and keeping myself from pleasure? This again is to no purpose, and a bitter work.
Two are better than one, because they have a good reward for their work.
And if one has a fall, the other will give him a hand; but unhappy is the man who is by himself, because he has no helper.
So again, if two are sleeping together they are warm, but how may one be warm by himself?

If I make use of the tongues of men and of angels, and have not love, I am like sounding brass, or a loud

На этом можно закрывать проект, ну всё выучил, молодец. И привёл тот чанк, с самым важным местом из писания, описывающее любовь.

## Протестировать на нескольких примерах, оченить качество



---



# Критерии оценки

Работа проверяется по следующим критериям (максимум 10 баллов):

### Загрузка и подготовка данных (2 балла)
- [ ] 0.5 балла: выбран критерий подбора материалов
- [ ] 0.5 балла: загружено не менее 100 записей/статей
- [ ] 0.5 балла: тексты успешно извлечены из источника
- [ ] 0.5 балла: данные приведены к формату, пригодному для чанкинга (очистка, объединение полей)

### Чанкинг (2 балла)
- [ ] 0.5 балла: выбран подходящий тип сплиттера (RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter и т.д.)
- [ ] 0.5 балла: обоснован выбор размера чанка и перекрытия (например, "512 токенов, overlap 20% для сохранения контекста")
- [ ] 0.5 балла: чанки созданы и не содержат явных артефактов (оборванных слов)
- [ ] 0.5 балла: количество чанков соответствует ожидаемому (не 1 и не 100500 на документ)

### Векторное хранилище (1 балл)
- [ ] 0.5 балла: выбрана адекватная эмбеддинг-модель (например, all-MiniLM-L6-v2 для русского/английского)
- [ ] 0.5 балла: индекс создан


### Реализация цепочки (3 балла)
- [ ] 0.5 балла: выбрана LLM
- [ ] 1 балл: промпт, QUESTION, TASK составлены корректно
- [ ] 0.5 балла: обоснован заданный TOP_K
- [ ] 0.5 балла: ответ генерируется на основе найденных чанков (видно по содержанию)
- [ ] 0.5 балла: обработан случай отсутствия информации в контексте

### Тестирование и анализ (2 балла)
- [ ] 0.5 балла: задано минимум 3 разнотипных вопроса (фактический, обобщающий, уточняющий)
- [ ] 0.5 балла: для каждого вопроса показан и проанализирован ответ
- [ ] 0.5 балла: в анализе указано, какие чанки использовались и почему
- [ ] 0.5 балла: сделан вывод о качестве работы системы (что получилось, что нет, гипотезы почему)
